In [1]:
import os
import numpy as np
import pandas as pd
from skimage.transform import rescale
from skimage.io import imread, imsave, imshow
from skimage.filters import gaussian
from tkinter import Tk
from tkinter import filedialog
import tifffile as tf
import dask
from skimage import io
import cv2 as cv

dask.config.set({"array.slicing.split_large_chunks": False})

fTypes = (".tif")                 

In [31]:
root = Tk()
root.withdraw()
strtPath = "/Users/nelsonlab/"
dirName = filedialog.askdirectory(initialdir=strtPath)

#print(dirName)
nucFold = dirName + "/nuc"
labFold = dirName + "/nuc/nLabs"
orNucFold = dirName + "/nuc/origSize"
rnaFold = dirName + "/rna"
if not os.path.exists(nucFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(nucFold)  
if not os.path.exists(labFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(labFold)  
if not os.path.exists(rnaFold):  
# Create directory for storing the tiff files for nuc channels 
   os.mkdir(rnaFold) 
if not os.path.exists(orNucFold):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(orNucFold) 

2024-06-17 15:37:32.309 python[2406:251644] +[CATransaction synchronize] called within transaction
2024-06-17 15:37:32.344 python[2406:251644] +[CATransaction synchronize] called within transaction
2024-06-17 15:37:33.960 python[2406:251644] +[CATransaction synchronize] called within transaction


In [10]:
# channels for cell, nucleus are hard coded, other chans are rna
n = 2
rna1 = 0
rna2 = 1

In [27]:
files = [f for f in os.listdir(dirName) if "P2C" in f]
filename = files[2]
print(filename)
base, ext = os.path.splitext(filename)
path = os.path.join(dirName, filename)
img = imread(path)                               # read the original image file
print("SHAPE= ",img.shape)
nuc_chan = img[0:2, :, :, n]
imsave("Documents/n1.tif", nuc_chan, check_contrast=False)
ds_nuc = rescale(nuc_chan, (1, 0.25, 0.25), anti_aliasing=True)
dsG = gaussian(ds_nuc, sigma=(0,1.3, 1.3), truncate = 3.5)
imsave("Documents/DSn1.tif", dsG, check_contrast=False) 

nuc_chan = img[2:4, :, :, n]
imsave("Documents/n2.tif", nuc_chan, check_contrast=False)
ds_nuc = rescale(nuc_chan, (1, 0.25, 0.25), anti_aliasing=True)
dsG = gaussian(ds_nuc, sigma=(0,1.3, 1.3), truncate = 3.5)
imsave("Documents/DSn2.tif", dsG, check_contrast=False) 

  
rna_chan1 = img[0:2, :, :, rna1]  
imsave("Documents/rna1_1.tif", rna_chan1, check_contrast=False)
rna_chan1 = img[2:4, :, :, rna1]  
imsave("Documents/rna1_2.tif", rna_chan1, check_contrast=False)
       

rna_chan2 = img[0:2, :, :, rna2]
imsave("Documents/rna2_1.tif", rna_chan2, check_contrast=False)
rna_chan2 = img[2:4, :, :, rna2]
imsave("Documents/rna2_2.tif", rna_chan2, check_contrast=False)

052824_P2C_K7K9_RH_crop.tif
SHAPE=  (4, 6624, 5256, 3)


In [65]:
z = np.concatenate((nuc_chan,blank), axis = 0)

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 3 dimension(s) and the array at index 1 has 2 dimension(s)

In [11]:
for filename in os.listdir(dirName):
   if filename.endswith(".tif") and not filename.startswith("."):
      print (filename)
      base, ext = os.path.splitext(filename)
      path = os.path.join(dirName, filename)
      img = imread(path)                               # read the original image file
      print("SHAPE= ",img.shape)
      if img.shape[0]==4:
           blank = np.zeros((img.shape[1],img.shape[2]))
#Save nucleus file          
      nfile = "nuc_" + base + ".tif"
      npath = os.path.join(orNucFold, nfile)
      nuc_chan = img[:, :, :, n]
      if img.shape[0]==4:
           nuc_chan[5,:,:] = blank
      imsave(npath, nuc_chan, check_contrast=False)

#Downscaling nucleus image for Cellpose       
      ds_nuc = rescale(nuc_chan, (1, 0.25, 0.25), anti_aliasing=True)
      dsG = gaussian(ds_nuc, sigma=(0,1.3, 1.3), truncate = 3.5)
      nfile = "DSnuc_" + base + ".tif"
      npath2 = os.path.join(nucFold,nfile)
      imsave(npath2, dsG, check_contrast=False)       
          
#Save RNA files        
      rnafile1 = "rna1_" + base + ".tif"
      rpath1 = os.path.join(rnaFold, rnafile1)  
      rna_chan1 = img[:, :, :, rna1]
      if img.shape[0]==4:
           rna_chan1[5,:,:] = blank
       
      imsave(rpath1, rna_chan1, check_contrast=False)
       
      rnafile2 = "rna2_" + base + ".tif"
      rpath2 = os.path.join(rnaFold, rnafile2)
      rna_chan2 = img[:, :, :, rna2]
      if img.shape[0]==4:
           rna_chan2[5,:,:] = blank
      imsave(rpath2, rna_chan2, check_contrast=False)

050824_ScrP10_GFP_K7K9_LH2_crop.tif
SHAPE=  (6, 8196, 6732, 3)
050824_ScrP20_GFP_K7K9_LH2_crop.tif
SHAPE=  (6, 7392, 5616, 3)
050824_ScrP10_GFP_K7K9_RH2_crop.tif
SHAPE=  (7, 7944, 4976, 3)
050824_ScrP20_GFP_K7K9_RH2_crop.tif
SHAPE=  (5, 7672, 4509, 3)


In [3]:
from cellpose import models
from cellpose import io as cpio

#cellPath = "/Users/sachanelson/programming/Jupyter/FISH/dataFromLiwei/cell"
#use_GPU = core.use_gpu()
#print('>>> GPU activated? %d'%use_GPU)

# call logger_setup to have output of cellpose written
from cellpose.io import logger_setup
logger_setup();

model = models.Cellpose(gpu=False, model_type='cyto')
channels = [0,0]

2024-06-10 19:35:48,905 [INFO] WRITING LOG OUTPUT TO /Users/nelsonlab/.cellpose/run.log
2024-06-10 19:35:48,906 [INFO] 
cellpose version: 	2.2.3 
platform:       	darwin 
python version: 	3.9.18 
torch version:  	2.1.2.post2
2024-06-10 19:35:48,906 [INFO] >>>> using CPU
2024-06-10 19:35:48,907 [INFO] >> cyto << model set to be used
2024-06-10 19:35:48,907 [INFO] WARNING: MKL version on torch not working/installed - CPU version will be slightly slower.
2024-06-10 19:35:48,907 [INFO] see https://pytorch.org/docs/stable/backends.html?highlight=mkl
2024-06-10 19:35:48,970 [INFO] >>>> model diam_mean =  30.000 (ROIs rescaled to this size during training)


In [33]:
for filename in (files):
   if filename.endswith(".tif"):
      fpath = os.path.join(nucFold, filename)
      outpath = os.path.join(labFold, filename)
      print("reading file",fpath)
      img = cpio.imread(fpath)
      if "40x" in filename:
          dset = 25
      else:
          dset = 12
      print("diameter = ",dset)
# might need to play with diameter, pre measure, or use the diameter=None setting
      masks, flows, styles, diams = model.eval(img, diameter=dset, cellprob_threshold=-3, do_3D=False, stitch_threshold=0.2)
      print("ran model")
#      fpath2 = cellPath + "/masks"
#      if not os.path.exists(fpath2):  
#         os.mkdir(fpath2)  
      cpio.save_masks(img, masks, flows, outpath, png=False, tif=True, save_flows=True)
   else:
      continue

reading file /Volumes/RWK_Imaging/p2p30/Good/cropped/nuc/DSnuc_040924_P2B_ProIII_RH_K7_K9_crop.tif
diameter =  12
2024-06-11 13:38:01,755 [INFO] ~~~ FINDING MASKS ~~~
2024-06-11 13:38:01,816 [INFO] multi-stack tiff read in as having 5 planes 1 channels
2024-06-11 13:39:55,680 [INFO] 100%|##########| 5/5 [01:53<00:00, 22.77s/it]
2024-06-11 13:39:55,681 [INFO] network run in 113.85s
2024-06-11 13:40:07,366 [INFO] stitching 5 planes using stitch_threshold=0.200 to make 3D masks
2024-06-11 13:40:07,461 [INFO] masks created in 11.78s
2024-06-11 13:40:07,611 [INFO] >>>> TOTAL TIME 125.86 sec
ran model


In [16]:
from skimage.segmentation import clear_border
from napari_simpleitk_image_processing import label_statistics
from skimage.measure import regionprops_table

sizecutoff = 2

upPath = os.path.join(labFold,"upscaled")
if not os.path.exists(upPath):  
# Create directory for storing the tiff files for cell channels 
   os.mkdir(upPath)

#Main loop - load in label/img files, filter out bad calls, save cell (img) properties, upscale label file
for filename in os.listdir(nucFold):
   if filename.endswith(".tif"):
       print(filename)
       filePath = os.path.join(nucFold, filename)
       labName = filename.replace(".tif","_cp_masks.tif")
       labPath = os.path.join(labFold, labName)
       if os.path.exists(labPath):
           img = io.imread(filePath)
           lab = io.imread(labPath)
           labfilt = lab.copy()
           regionprops = label_statistics(img,lab, True, True, True, True)
       #Find and filter artifacts (bottom x% of size) 
           cutoff = ((regionprops["number_of_pixels"].max() - regionprops["number_of_pixels"].min())/100) * sizecutoff
           cellidx = regionprops[regionprops["number_of_pixels"] < cutoff]["label"].to_numpy()
           
           labfilt[np.where(np.isin(labfilt, cellidx))] = 0
           
           if labName.endswith("_cp_masks.tif") and labName.startswith("DS"):
               lab_up = rescale(labfilt, (1, 4, 4), anti_aliasing=False, preserve_range=True, order=0)
               c4fname = labName[2:]
               c4fpath = os.path.join(upPath, c4fname)
               imsave(c4fpath, lab_up, check_contrast=False)
               print("saved rescaled filtered labels")
           else:
               print ("wrong filetype--not DS cp_mask", filename)
               continue
       else:
           print("label file not found")
           continue


DSnuc_050824_ScrP10_GFP_K7K9_LH2_crop.tif
saved rescaled filtered labels
DSnuc_050824_ScrP10_GFP_K7K9_RH2_crop.tif
saved rescaled filtered labels
DSnuc_050824_ScrP20_GFP_K7K9_LH2_crop.tif
saved rescaled filtered labels
DSnuc_050824_ScrP20_GFP_K7K9_RH2_crop.tif
saved rescaled filtered labels


In [32]:
import matplotlib
import numpy as np
import os

import pyclesperanto_prototype as cle
cle.select_device("Apple M1 Max")

rlabFold = dirName + "/rna/rLabs"
if not os.path.exists(rlabFold):  
   os.mkdir(rlabFold)  
files = os.listdir(rnaFold)

files

['rna1_061024_P2A_K7K9_LHD40x_crop.tif',
 'rna2_061024_P2A_K7K9_LHD40x_crop.tif',
 'rna1_061024_P2A_K7K9_LHU40x_crop.tif',
 'rna2_061024_P2A_K7K9_LHU40x_crop.tif',
 'rna1_061024_P2A_K7K9_RHD40x_crop.tif',
 'rna2_061024_P2A_K7K9_RHD40x_crop.tif',
 'rna1_061024_P2A_K7K9_RHU40x_crop.tif',
 'rna2_061024_P2A_K7K9_RHU40x_crop.tif',
 'rna1_061024_P7A_K7K9_LH_crop.tif',
 'rna2_061024_P7A_K7K9_LH_crop.tif',
 'rna1_061024_P7A_K7K9_RH_crop.tif',
 'rna2_061024_P7A_K7K9_RH_crop.tif',
 'rna1_061024_P7D_K7K9_LH_crop.tif',
 'rna2_061024_P7D_K7K9_LH_crop.tif',
 'rna1_061024_P7D_K7K9_RH_crop.tif',
 'rna2_061024_P7D_K7K9_RH_crop.tif',
 'rna1_061024_P30C_K7K9_LH_crop.tif',
 'rna2_061024_P30C_K7K9_LH_crop.tif',
 'rna1_052024_P2D_K9K7_RH_crop.tif',
 'rna2_052024_P2D_K9K7_RH_crop.tif',
 'rLabs']

In [33]:
for filename in files:
   if filename.endswith(".tif") and "40x" not in filename:
      print (filename)
      rPath = os.path.join(rnaFold, filename)
      rnaIn = imread(rPath)
      if rnaIn.shape[0] > 100:
           rnaIn = rnaIn.transpose(2,0,1)
      print(rnaIn.shape)
      labOut = np.zeros(shape=rnaIn.shape, dtype=np.int32)
      base, ext = os.path.splitext(filename)
      rLabFile = base + "_lab.tif"
      rLabPath = os.path.join(rlabFold,rLabFile)
#   print("base=",base," ext=", ext)
      rna = cle.push(rnaIn)
      seg = cle.voronoi_otsu_labeling(rna, spot_sigma = 2, outline_sigma=1)
             
      imsave(rLabPath, seg, check_contrast=False)

rna1_061024_P7A_K7K9_LH_crop.tif
(5, 6696, 5208)
rna2_061024_P7A_K7K9_LH_crop.tif
(5, 6696, 5208)
rna1_061024_P7A_K7K9_RH_crop.tif
(5, 6248, 6168)
rna2_061024_P7A_K7K9_RH_crop.tif
(5, 6248, 6168)
rna1_061024_P7D_K7K9_LH_crop.tif
(5, 6504, 5656)
rna2_061024_P7D_K7K9_LH_crop.tif
(5, 6504, 5656)
rna1_061024_P7D_K7K9_RH_crop.tif
(5, 6592, 5520)
rna2_061024_P7D_K7K9_RH_crop.tif
(5, 6592, 5520)
rna1_061024_P30C_K7K9_LH_crop.tif
(5, 8184, 6408)
rna2_061024_P30C_K7K9_LH_crop.tif
(5, 8184, 6408)
rna1_052024_P2D_K9K7_RH_crop.tif
(6, 4686, 4484)
rna2_052024_P2D_K9K7_RH_crop.tif
(6, 4686, 4484)


In [36]:
rlabFold = dirName + "/rna/rLabs"
tabFold = dirName + "/table"
if not os.path.exists(tabFold):  
   os.mkdir(tabFold)  

rLabFiles = os.listdir(rlabFold)
nLabfiles = os.listdir(upPath)

print(nLabfiles)

['nuc_061024_P30C_K7K9_LH_crop_cp_masks.tif', 'nuc_061024_P2A_K7K9_LHD40x_crop_cp_masks.tif', 'nuc_061024_P2A_K7K9_RHD40x_crop_cp_masks.tif', 'nuc_061024_P2A_K7K9_RHU40x_crop_cp_masks.tif', 'nuc_061024_P7A_K7K9_LH_crop_cp_masks.tif', 'nuc_061024_P7A_K7K9_RH_crop_cp_masks.tif', 'nuc_061024_P7D_K7K9_LH_crop_cp_masks.tif', 'nuc_061024_P7D_K7K9_RH_crop_cp_masks.tif', 'nuc_052024_P2D_K9K7_RH_crop_cp_masks.tif']


In [37]:
verbose=True

for filename in nLabfiles:
   if filename.endswith("_cp_masks.tif") and "40x" not in filename:
      print (filename)
      filename_rna = filename.replace("_cp_masks.tif","_lab.tif")
      nFilePath = os.path.join(upPath,filename)
      nLabs = imread(nFilePath)
      print("Reading: ",nFilePath)
      ncells = np.shape(np.unique(nLabs))[0]
      tempdf=np.zeros((nLabs.shape[1],3))
      n2 = 0
      for n in range(nLabs.shape[1]):
          tempdf[n,n2]=sum(np.unique(nLabs[:,n,:])!=0)
      if verbose: 
          print("Cell shapes=",nLabs.shape) 
          print("#cells=",ncells)
      for rLabfname in rLabFiles:
          if rLabfname.endswith(filename_rna[4:]):
              n2 = n2 + 1
              rLabPath = os.path.join(rlabFold,rLabfname)         
              rseg = imread(rLabPath)
              print("Reading: ", rLabPath)
              if rseg.shape[0] > 100:
                  rseg = rseg.transpose(2,0,1)
              if verbose: 
                  print("Rna shapes=",rseg.shape)
                  numRNA = np.max(rseg)
                  print("#rna=", numRNA)
              for n in range(nLabs.shape[1]):
                  tempdf[n,n2]=sum(np.unique(rseg[:,n,:])!=0)
      TabFile = filename[4:].replace("cp_masks.tif","punctacounts.csv")
      TabPath = os.path.join(tabFold,TabFile)
      outdf = pd.DataFrame(data = tempdf,  columns = ["nNuc","nRNA1","nRNA2"])
      outdf.to_csv(TabPath)


nuc_061024_P30C_K7K9_LH_crop_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/nuc/nLabs/upscaled/nuc_061024_P30C_K7K9_LH_crop_cp_masks.tif
Cell shapes= (5, 8184, 6408)
#cells= 3479
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/rna/rLabs/rna1_061024_P30C_K7K9_LH_crop_lab.tif
Rna shapes= (5, 8184, 6408)
#rna= 106189
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/rna/rLabs/rna2_061024_P30C_K7K9_LH_crop_lab.tif
Rna shapes= (5, 8184, 6408)
#rna= 40059
nuc_061024_P7A_K7K9_LH_crop_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/nuc/nLabs/upscaled/nuc_061024_P7A_K7K9_LH_crop_cp_masks.tif
Cell shapes= (5, 6696, 5208)
#cells= 3954
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/rna/rLabs/rna1_061024_P7A_K7K9_LH_crop_lab.tif
Rna shapes= (5, 6696, 5208)
#rna= 64004
Reading:  /Volumes/RWK_Imaging/p2p30/Good/set2/rna/rLabs/rna2_061024_P7A_K7K9_LH_crop_lab.tif
Rna shapes= (5, 6696, 5208)
#rna= 62768
nuc_061024_P7A_K7K9_RH_crop_cp_masks.tif
Reading:  /Volumes/RWK_Imaging/p2p30/G